[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/smolagents/smolagents_prompt_chaining.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-1px8dcmlf-fmThhDFD_V_48oU7ALan4Q">社区</a>
    </p>
</center>

# SmolAgents 提示词链智能体

本教程介绍如何使用 [Smolagents](https://huggingface.co/docs/smolagents/index) 构建**提示词链**，并通过 Phoenix 进行追踪。
提示词链将复杂任务拆分为相互衔接的小任务，前一步的输出成为下一步的输入。它适用于能够明确拆分为固定子任务的场景，以增加延迟为代价，让每次 LLM 调用更简单、更准确，从而改善整体表现。

本例构建多步骤简历分析流程，从提取关键词到总结分析结果，全程自动执行并可追踪。

完成本教程后，你将学会：

- 设置专业 Smolagents 智能体
- 在智能体之间传递上下文，串联任务
- 通过简化每次 LLM 调用提高任务准确性
- 使用 Phoenix 追踪并可视化完整执行过程

⚠️ 本教程需要 Hugging Face 令牌。

## 设置密钥与依赖

In [ ]:
!pip install openinference-instrumentation-smolagents smolagents

In [ ]:
import os
from getpass import getpass

from phoenix.otel import register
from smolagents import CodeAgent, HfApiModel, tool

if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("🔑 请输入 Hugging Face 令牌： ")

if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

## 配置模型

In [ ]:
# 初始化模型
model = HfApiModel()

## 配置追踪

In [ ]:
tracer_provider = register(
    project_name="smolagents-prompt-chaining",
    protocol="http/protobuf",
    auto_instrument=True,
)

## 提示词链示例任务：简历分析

本例演示如何使用 Smolagents 提示词链分析简历。流程先分析简历中的关键词、技能与经历，再总结分析结果。

## 定义智能体

In [ ]:
# 定义 extract_keywords 工具


@tool
def extract_keywords(text: str) -> str:
    """
    从输入文本中提取关键词。

    Args:
        text: 要从中提取关键词的简历文本。

    Returns:
        以逗号分隔、长度超过 5 个字母的关键词字符串。
    """
    words = text.split()
    keywords = [word.strip(".,") for word in words if len(word) > 5]
    return ", ".join(keywords)


# 使用自定义工具创建智能体
agent = CodeAgent(tools=[extract_keywords], model=model)

# 使用串联推理步骤的提示词运行智能体
agent.run(
    "从这份简历提取关键词： 'Experienced software engineer with 5+ years in Python, "
    "React, and cloud technologies. Led development of microservices architecture and implemented "
    "CI/CD pipelines. Strong background in machine learning and data analysis.' "
    "然后总结这些关键词反映的候选人技术专长。"
)